In [1]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler

In [2]:
esa_21_22_fast_actions_parsed = pd.read_pickle('esa_21_22_fast_actions_parsed.pkl')
esa_22_23_fast_actions_parsed = pd.read_pickle('esa_22_23_fast_actions_parsed.pkl')
len(esa_22_23_fast_actions_parsed)

20632

In [3]:
euro_fast_actions_parsed = pd.read_pickle('euro_fast_actions_parsed.pkl')
fifa_fast_actions_parsed = pd.read_pickle('fifa_fast_actions_parsed.pkl')

In [4]:
esa_22_23_core = pd.read_pickle('38_235.pkl')

In [7]:
def get_test_set(df, df_core):
    dfs_test = []
    dfs_train = []
    teams = list(set(df['team'].values))
    test_matches = df_core[esa_22_23_core['match_week'] >= 18]['match_id'].values
    train_matches = df_core[esa_22_23_core['match_week'] < 18]['match_id'].values
    for team in teams:
        for match_id in test_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_test.append(df_result)
        for match_id in train_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_train.append(df_result)
    df_test = pd.concat(dfs_test)
    df_train = pd.concat(dfs_train)
    return df_test, df_train


esa_22_23_test, esa_22_23_train = get_test_set(esa_22_23_fast_actions_parsed, esa_22_23_core)
print(len(esa_22_23_test))
print(len(esa_22_23_train))

10847
9785


In [8]:
test_set = esa_22_23_test
train_set = pd.concat([esa_22_23_train, esa_21_22_fast_actions_parsed, euro_fast_actions_parsed, fifa_fast_actions_parsed])
print(len(test_set))
len(train_set)

10847


38522

In [9]:
test_set['was_shot'] = [1 if (shots > 0) else 0 for shots in test_set['num_of_shot'].values]
negative_test = len(test_set[test_set['was_shot'] == 0])
positive_test = len(test_set[test_set['was_shot'] == 1])
print(negative_test)
positive_test


10462


385

In [10]:
train_set['was_shot'] = [1 if (shots > 0) else 0 for shots in train_set['num_of_shot'].values]
negative_train = len(train_set[train_set['was_shot'] == 0])
positive_train = len(train_set[train_set['was_shot'] == 1])
print(negative_train)
positive_train

37187


1335

In [11]:
negative_total = negative_test + negative_train
positive_total = positive_test + positive_train
data_balance = negative_total / positive_total
data_balance

27.702906976744185

In [12]:
def get_sets_for_model(df_train, df_test):
    df_train = shuffle(df_train)
    df_test = shuffle(df_test)

    # scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))

    # df_train_ids = df_train['id'].values  # type: ignore
    # df_test_ids = df_test['id'].values  # type: ignore
    # df_train_teams = df_train['team'].values  # type: ignore
    # df_test_teams = df_test['team'].values  # type: ignore
    # df_train_was_shot = df_train['was_shot'].values  # type: ignore
    # df_test_was_shot = df_test['was_shot'].values  # type: ignore
    # # df_train_good_ends = df_train['good_end'].values  # type: ignore
    # # df_test_good_ends = df_test['good_end'].values  # type: ignore

    # df_train = df_train.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'was_shot']) # type: ignore
    # df_test = df_test.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'was_shot']) # type: ignore

    df_train_columns = df_train.columns.sort_values()
    df_test_columns = df_test.columns.sort_values()

    # df_train = scaler.fit_transform(df_train)
    # df_test = scaler.fit_transform(df_test)

    df_train = pd.DataFrame(df_train, columns=df_train_columns)
    df_test = pd.DataFrame(df_test, columns=df_test_columns)

    # df_train['id'] = df_train_ids
    # df_test['id'] = df_test_ids
    # df_train['team'] = df_train_teams
    # df_test['team'] = df_test_teams
    # df_train['was_shot'] = df_train_was_shot
    # df_test['was_shot'] = df_test_was_shot

    y_train = df_train['was_shot']
    X_train = df_train.drop(columns=['was_shot'])

    y_test = df_test['was_shot']
    X_test = df_test.drop(columns=['was_shot'])

    return X_train, X_test, y_train, y_test

In [13]:
X_train, X_test, y_train, y_test = get_sets_for_model(train_set, test_set)

In [14]:
from xgboost import XGBClassifier

def train_model(X_train, X_test, y_train):
    model = XGBClassifier(objective='binary:logistic', scale_pos_weight=data_balance)

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    model.fit(input_X_train, y_train)

    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    XGB_predictions = model.predict(input_X_test)

    return model, XGB_predictions, input_X_test

model, pred, input_test = train_model(X_train, X_test, y_train)

In [15]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def evaluate_model(y_test, pred):
    print('Confusion matrix')
    print(confusion_matrix(y_test, pred))
    print('Classification report')
    print(classification_report(y_test, pred))
    print('Accuracy= ', accuracy_score(y_test, pred))

evaluate_model(y_test, pred)

Confusion matrix
[[10418    44]
 [   27   358]]
Classification report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     10462
           1       0.89      0.93      0.91       385

    accuracy                           0.99     10847
   macro avg       0.94      0.96      0.95     10847
weighted avg       0.99      0.99      0.99     10847

Accuracy=  0.9934544113579792


In [16]:
from datetime import datetime

# datetime object containing current date and time
now = datetime.now()
 
print("now =", now)

# dd/mm/YY H:M:S
dt_string = now.strftime("%d_%m_%Y_%H-%M-%S")
print("date and time =", dt_string)

now = 2023-06-13 22:45:25.744532
date and time = 13_06_2023_22-45-25


In [17]:
def feature_importances_raport(input_X_test, model, set_title):
    result = pd.DataFrame(columns=['name', 'value'])
    result['name'] = input_X_test.columns
    result['value'] = model.feature_importances_

    result = result.sort_values(by=['value'], ascending=False)

    result.to_csv(f'features_{set_title}.csv')
    print(result)
    return result

feature_importances = feature_importances_raport(input_test, model, f'fast_sequences_experiment_{dt_string}')

                    name     value
1                  end_x  0.509095
2                  end_y  0.071674
5   num_of_ball_receipt*  0.061543
10   num_of_dispossessed  0.060245
16     num_of_miscontrol  0.059484
14         num_of_events  0.035978
17           num_of_pass  0.024778
8           num_of_carry  0.024440
6   num_of_ball_recovery  0.021011
18    opponent_distances  0.017572
15   num_of_interception  0.016515
7           num_of_block  0.014835
12           num_of_duel  0.014830
11        num_of_dribble  0.011444
19               start_x  0.010362
20               start_y  0.010056
9       num_of_clearance  0.009619
0               duration  0.009357
3           key_pass_len  0.008632
4               match_id  0.008532
13          num_of_error  0.000000


In [18]:
def generate_scores(model, X_test, X_train, y_train, set_title):
    model = XGBClassifier(objective='binary:logistic', scale_pos_weight=data_balance)

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    model.fit(input_X_train, y_train)
    
    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    model = model.predict_proba(input_X_test)

    X_test['score_positive'] = [m[1] for m in model]
    X_test['score_negative'] = [m[0] for m in model]

    X_test.to_csv(f'{set_title}_evaluated.csv')

    return X_test

df_with_scores = generate_scores(model, X_test, X_train, y_train, f'fast_sequences_experiment_{dt_string}')

In [19]:
len(test_set[(test_set['team'] == 'Lech Poznan') & (test_set['num_of_shot'] > 0)])

26

In [20]:
import statistics
import pandas as pd

def top_teams_by(df_with_scores, by, dt_string):
    teams = list(set(df_with_scores['team'].values))
    df_team_scores = pd.DataFrame()
    df_scores = []
    df_teams = []
    df_obv = []
    df_xg = []
    df_sequences = []
    df_shots = []
    for team in teams:
        df_with_scores_team = df_with_scores[df_with_scores['team'] == team]
        if by == 'median':
            score = statistics.median(df_with_scores_team['score_positive'].values)
            obv = statistics.median(df_with_scores_team['obv_for'].values)
            xg = statistics.median(df_with_scores_team['xg'].values)
        elif by == 'mean':
            score = statistics.mean(df_with_scores[df_with_scores['team'] == team]['score_positive'].values)
            obv = statistics.mean(df_with_scores[df_with_scores['team'] == team]['obv_for'].values)
            xg = statistics.mean(df_with_scores[df_with_scores['team'] == team]['xg'].values)
        else:
            score = sum(df_with_scores[df_with_scores['team'] == team]['score_positive'])
            obv = sum(df_with_scores[df_with_scores['team'] == team]['obv_for'])
            xg = sum(df_with_scores[df_with_scores['team'] == team]['xg'])
        sequences = len(df_with_scores_team)
        shots = len(df_with_scores_team[df_with_scores_team['num_of_shot'] > 0])
        df_teams.append(team)
        df_obv.append(obv)
        df_xg.append(xg)
        df_scores.append(score)
        df_sequences.append(sequences)
        df_shots.append(shots)
    df_team_scores['team'] = df_teams
    df_team_scores['score'] = df_scores
    df_team_scores['obv'] = df_obv
    df_team_scores['xg'] = df_xg
    df_team_scores['sequences'] = df_sequences
    df_team_scores['shots'] = df_shots
    result = df_team_scores.sort_values(by=['score'], ascending=False)
    result.to_csv(f'fast_sequences_top_teams_{by}_{dt_string}.csv')
    return result

In [21]:
top_teams_by(df_with_scores, 'sum', dt_string)

,team,score,obv,xg,sequences,shots
14,Pogoń Szczecin,37.640254,6.532318,3.782754,555,36
17,Zagłębie Lubin,35.910069,5.063947,3.965049,607,34
16,Piast Gliwice,28.678021,3.652952,1.964929,693,29
5,Lech Poznan,28.660708,5.395132,3.401627,505,26
15,Jagiellonia Białystok,27.518151,4.591142,2.457675,658,24
1,RKS Radomiak Radom,27.006463,5.449739,2.254996,759,24
13,Śląsk Wrocław,26.760923,4.197969,2.850235,614,27
12,Cracovia Kraków,25.789103,3.666251,2.909048,642,23
4,Widzew Łódź,24.267227,4.480748,1.795778,559,19
8,Górnik Zabrze,22.927964,2.964828,1.644435,601,21


In [22]:
top_teams_by(df_with_scores, 'mean', dt_string)

,team,score,obv,xg,sequences,shots
14,Pogoń Szczecin,0.067820,0.011770,0.006816,555,36
17,Zagłębie Lubin,0.059160,0.008343,0.006532,607,34
5,Lech Poznan,0.056754,0.010683,0.006736,505,26
2,Legia Warszawa,0.046804,0.006706,0.002873,471,17
13,Śląsk Wrocław,0.043585,0.006837,0.004642,614,27
4,Widzew Łódź,0.043412,0.008016,0.003212,559,19
15,Jagiellonia Białystok,0.041821,0.006977,0.003735,658,24
16,Piast Gliwice,0.041382,0.005271,0.002835,693,29
12,Cracovia Kraków,0.040170,0.005711,0.004531,642,23
8,Górnik Zabrze,0.038150,0.004933,0.002736,601,21


In [23]:
top_teams_by(df_with_scores, 'median', dt_string)

,team,score,obv,xg,sequences,shots
2,Legia Warszawa,0.000034,0.001565,0.0,471,17
5,Lech Poznan,0.000033,0.001365,0.0,505,26
7,Raków Częstochowa,0.000032,0.001454,0.0,576,13
14,Pogoń Szczecin,0.000030,0.001545,0.0,555,36
8,Górnik Zabrze,0.000030,0.001726,0.0,601,21
1,RKS Radomiak Radom,0.000030,0.001832,0.0,759,24
4,Widzew Łódź,0.000030,0.001457,0.0,559,19
17,Zagłębie Lubin,0.000029,0.001917,0.0,607,34
15,Jagiellonia Białystok,0.000029,0.001955,0.0,658,24
9,Wisła Płock,0.000028,0.001769,0.0,550,10


In [32]:
def show_correlation(df_with_scores, dt_string):
    corr_with_xg = df_with_scores['score_positive'].corr(df_with_scores['xg'])
    corr_with_obv = df_with_scores['score_positive'].corr(df_with_scores['obv_for'])
    corr_with_obv_total = df_with_scores['score_positive'].corr(df_with_scores['obv_total'])
    print(f'Correlation with xg: {corr_with_xg:.3f}')
    print(f'Correlation with obv_for: {corr_with_obv:.3f}')
    print(f'Correlation with obv_total: {corr_with_obv_total:.3f}')
    result = pd.DataFrame()
    result['xg_corr'] = [corr_with_xg]
    result['obv_corr'] = [corr_with_obv]
    result['obv_total'] = [corr_with_obv_total]
    result.to_csv(f'fast_sequences_correlations_{dt_string}.csv')

In [33]:
show_correlation(df_with_scores, dt_string)

Correlation with xg: 0.612
Correlation with obv_for: 0.368
Correlation with obv_total: 0.368


In [29]:
test_set

,id,team,duration,num_of_pass,num_of_carry,num_of_miscontrol,num_of_dribble,num_of_error,num_of_duel,num_of_ball_receipt*,...,start_y,end_x,end_y,key_pass_len,opponent_distances,match_id,obv_for,obv_total,xg,was_shot
0,4ae07a8a-2412-474d-8eae-987969ca6d0d#c3d99c3f-...,Górnik Zabrze,16,6,4,0,0,0,0,6,...,40.0,78.3,18.5,31.500635,23.866954,3836470,0.004016,0.004016,0.0,0
0,9b82ca02-17d0-4d45-ba1c-2f7d3f1d8e77#be513c6d-...,Górnik Zabrze,102,1,0,0,0,0,0,1,...,32.9,67.4,24.7,25.314225,16.404687,3836470,-0.000761,-0.000761,0.0,0
0,4bf03327-34e2-4bd7-92a4-c57ed92999ff#08628d87-...,Górnik Zabrze,47,1,1,0,0,0,0,1,...,54.9,63.4,46.2,30.203477,15.650754,3836470,0.000882,0.000882,0.0,0
0,16f2632e-7e5c-46f2-b047-1a557f5e49cf#17b72ce4-...,Górnik Zabrze,28,2,0,0,0,0,0,2,...,44.0,74.5,56.3,58.860003,12.633143,3836470,0.003628,0.003628,0.0,0
0,1c136e0c-3eed-43f8-ba56-adf044d4bd10#081e5d4c-...,Górnik Zabrze,69,3,4,0,0,0,1,3,...,76.4,79.2,13.8,0.000000,15.726425,3836470,0.001899,0.001899,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,06bcb7e8-a44a-4d58-9281-f0b6bea69fbc#77639ab6-...,Zagłębie Lubin,3,1,0,0,0,0,0,1,...,43.9,61.1,71.0,72.254066,0.000000,3836375,0.001572,0.001572,0.0,0
0,eae5183e-36cf-4482-899f-12f8c31f8135#b72b6f18-...,Zagłębie Lubin,110,1,1,0,0,0,0,1,...,27.6,65.0,3.7,0.000000,20.821588,3836375,0.000008,0.000008,0.0,0
0,ee1e5d5c-94bc-4947-a1ca-43fef132cfe1#c42f2c0a-...,Zagłębie Lubin,41,1,0,0,0,0,0,1,...,0.0,75.0,14.0,17.142054,15.506551,3836375,-0.000402,-0.000402,0.0,0
0,d732047d-aef4-43f0-9747-078f4103806f#41cdb8c1-...,Zagłębie Lubin,97,0,1,0,1,0,0,0,...,54.6,64.7,67.6,0.000000,17.982269,3836375,0.005964,0.005964,0.0,0


In [31]:
df_with_scores

,duration,end_area,end_x,end_y,id,key_pass_len,match_id,num_of_ball_receipt*,num_of_ball_recovery,num_of_block,...,obv_total,opponent_distances,path,start_area,start_x,start_y,team,xg,score_positive,score_negative
0,22,K5,107.5,31.6,e6b3f24c-a624-4168-ac7d-3b890889897f#9b6d63f7-...,68.740090,3836405,4,0,0,...,0.004344,14.085416,"[A4, H3, G3, H3, J2, K2, K1, K4, K5]",A4,7.0,43.9,Warta Poznań,0.0,3.309155e-01,0.669085
0,59,L6,117.5,20.5,c45268e9-be7d-4291-957d-f645260abc98#59bc08e3-...,28.093060,3836392,1,1,0,...,0.000132,16.312320,"[F8, J7, L6, L7, L6]",F8,59.2,2.4,Warta Poznań,0.0,3.347076e-02,0.966529
0,96,H4,73.4,48.5,5a30fff6-a9a0-4ed6-a52b-f715db99fb3c#fbfb02d6-...,7.306162,3836328,1,1,0,...,0.008662,18.471034,"[F2, F3, H3, H4]",F2,57.0,61.5,RKS Radomiak Radom,0.0,1.566927e-05,0.999984
0,40,G7,62.9,12.7,8e442e2d-0b8e-4c3f-a07e-fc7f6edc3f7b#45eb7816-...,9.786726,3836434,3,0,0,...,-0.001772,11.673156,"[F7, G7, H8, G8, G7]",F7,55.4,12.7,Piast Gliwice,0.0,8.571657e-06,0.999991
0,73,G6,64.2,23.6,8e12ee11-4a6b-4790-afe7-3fe126bd0175#51060f27-...,58.507521,3836346,1,0,0,...,0.003349,21.855412,"[A5, G6]",A5,7.0,35.9,Jagiellonia Białystok,0.0,2.049778e-05,0.999979
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,104,H8,73.4,8.4,c6bf56c2-850e-4d0e-9e0f-7fc20a05824b#d4ce13e3-...,22.720035,3836464,3,0,0,...,0.002102,15.770875,"[B8, E8, E6, F6, H7, H8]",B8,19.5,7.1,Piast Gliwice,0.0,1.261420e-05,0.999987
0,50,H7,72.7,10.4,e1b6aaa2-a1ff-482b-8cf9-1dfbbce1c252#b0029ff2-...,25.424398,3836407,1,0,0,...,0.008106,14.557185,"[E9, H7]",E9,49.5,0.0,Miedź Legnica,0.0,1.035797e-05,0.999990
0,104,J7,96.1,16.7,d5a73e25-ba80-4cb6-a340-d1b1fc64a2a5#f7f0e3ba-...,43.908997,3836390,1,1,0,...,0.004036,14.555888,"[E5, F5, J7]",E5,40.2,34.5,Legia Warszawa,0.0,4.077002e-07,1.000000
0,22,H8,74.1,9.1,da6ccaf3-6301-4d53-837d-d2ffa7414ca0#bb94760b-...,0.000000,3836464,1,0,0,...,0.000513,19.175165,"[B2, E1, H8]",B2,19.5,62.1,Warta Poznań,0.0,9.509493e-06,0.999990


In [30]:
df_with_scores.to_pickle('df_with_scores.pkl')
df_with_scores['id'].values

array(['e6b3f24c-a624-4168-ac7d-3b890889897f#9b6d63f7-e0a3-49c9-a9f1-09412258919a',
       'c45268e9-be7d-4291-957d-f645260abc98#59bc08e3-fa98-4f24-a591-801b1f756a79',
       '5a30fff6-a9a0-4ed6-a52b-f715db99fb3c#fbfb02d6-21e1-4eee-bcad-f3b8c45f031e',
       ...,
       'd5a73e25-ba80-4cb6-a340-d1b1fc64a2a5#f7f0e3ba-1a01-4c64-aea5-07e5e0bc8da4',
       'da6ccaf3-6301-4d53-837d-d2ffa7414ca0#bb94760b-c01c-401b-a2dc-7b06325eafe9',
       'f102f7ce-ec2d-4102-8344-f65833b7ab0e#bde65df5-e5cc-404c-95ed-f86997ad17bb'],
      dtype=object)